This notebook is for generating stats (such as chi squared and modulation index) for a set of measurements.
An example set of measurements might be a dataframe of pulsar measurements from VAST.
This notebook should be run on raw pulsar measurements.

IN: 'all_pulsar_measurements.csv', 'paper_dfv0.csv'

OUT: 'paper_dfv1.csv'

In [1]:
# importing required modules
import matplotlib.pyplot as plt
import pandas as pd

from astropy import units as u

import numpy as np

from scipy.stats import chi2

In [2]:
# setting plot preferences
%matplotlib inline

In [3]:
#computes how many times a source has been detected in VAST
def num_detections(source_df):
    bools = pd.isna(source_df['flux_peak'])
    counter = 0
    for item in bools:
        if not item:
            counter += 1
    return counter

In [4]:
# calculates a weighted mean of flux density for sources
def weighted_mean_flux_density(source_df):
    sum_1overvariance = 0
    for i in np.arange(source_df.shape[0]):
        if (not np.isnan(source_df['flux_peak'].iloc[i])):
            oneOverVariance = 1 / source_df['rms_image'].iloc[i]
            sum_1overvariance += (oneOverVariance ** 2)
    sumSovervariance = 0
    for i in np.arange(source_df.shape[0]):
        if (not np.isnan(source_df['flux_peak'].iloc[i])):
            S = source_df['flux_peak'].iloc[i]
            var = source_df['rms_image'].iloc[i] **2
            sumSovervariance += (S/var)
    err = 1 / (sum_1overvariance**(0.5))
    return (sumSovervariance / sum_1overvariance, err)       

In [5]:
# calculates chi_squared of a single source
def chi_squared(source_df):
    n_epochs = 0
    (S_weighted, S_weighted_err) = weighted_mean_flux_density(source_df)
    temp = 0
    for i in np.arange(source_df.shape[0]):
        if (not np.isnan(source_df['flux_peak'].iloc[i])):
            S = source_df['flux_peak'].iloc[i]
            var = source_df['rms_image'].iloc[i] ** 2
            num = (S - S_weighted)**2
            temp += num / var
            n_epochs += 1
            
    source_df = source_df[source_df['detection']==True]
    fluxes = source_df['flux_peak']
    errs = source_df['rms_image']
    
    err = 2 * ((fluxes - S_weighted).pow(2).div(errs.pow(4)).mul(errs.pow(2) + S_weighted_err**2)).sum()**(0.5)
        
    return (temp, n_epochs, err)

In [6]:
# calculates modulation index of a single source
def calc_modulation(source_df):
    source_df = source_df[source_df['detection']==True]
    fluxes = source_df['flux_peak']
    errs = source_df['rms_image']
    mean = source_df['flux_peak'].mean()
    #mean_err = errs.mean()
    std = source_df['flux_peak'].std()
    N = source_df.shape[0]
    mean_err = std / (N**(0.5))
    std_err = (1 / (2 *(N - 1)))**(0.5) * std
    
    mod = 100 * (std / mean)
    #err = ([(fluxes - mean).pow(2).mul((errs.pow(2) + mean_err**2)).sum() / (N**2) / (std**4)] + (mean_err / mean)**2)**(0.5) * mod * 100
    #err = (((fluxes - mean) / std - std).mul(errs).sum() / N / (mean**2) * 100)
    #err = abs(err)
    err = ((std_err / std)**2 + (mean_err / mean)**2)**(0.5) * mod
    
    return (mod, err)

In [43]:
psrs_df = pd.read_csv('paper_dfv0.csv')

In [8]:
data = pd.read_csv('all_pulsar_measurements.csv')

<ipython-input-8-76d482e411e9>:1: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv('all_psr_measurements_corrected.csv')


In [12]:
#editing data so that certain columns reflect correct values
#this cell may need to be modified depending on changes to your data
data = data[data['freq']!=1367.5]

data['detection'] = data['flux_peak'].notna()

data.rename(columns={data.columns[0]: 'psr_name'}, inplace=True)

data.drop(columns=data.columns[0], inplace=True)

In [26]:
#computing stats for each pulsar. The lists below this line are containers for these stats
mod = []
mod_err = []
chi = []
chi_reduced = []
p_val = []
dof = []
for name in psrs_df['JNAME']:
    print(name)
    psr_meas = data[data['psr_name']==name]
    
    if (psr_meas[psr_meas['detection']].shape[0]<2): #if this source has less than 2 detections
        (mod_val, mod_err_val) = (np.nan, np.nan)
        (chi_val, n, _) = (np.nan, np.nan, np.nan)
        p = np.nan
    else:
        (mod_val, mod_err_val) = calc_modulation(psr_meas)
        (chi_val, n, _) = chi_squared(psr_meas)
        p = chi2.sf(chi_val, n-1)
        
    mod.append(mod_val)
    mod_err.append(mod_err_val)
    
    chi.append(chi_val)
    chi_reduced.append(chi_val / (n - 1))
    
    p_val.append(p)
    
    dof.append(n-1)

J1644-4559
J1752-2806
J0908-4913
J1534-5334
J1745-3040
J1605-5257
J1804-2717
J1604-4909
J1428-5530
J1001-5507
J1722-3207
J1806-1154
J0907-5157
J1717-4054
J1801-1417
J1359-6038
J1809-1943
J1550-5418
J1701-3726
J1709-4429
J1302-6350
J1829-1751
J1720-2933
J1549-4848
J0955-5304
J1740-3015
J1042-5521
J0942-5657
J1651-4246
J1801-2920
J1600-5751
J1544-5308
J1816-2650
J1835-1106
J1110-5637
J1753-1914
J1804-2858
J1836-1008
J1759-2205
J1803-2137
J1811-2405
J1539-5626
J1633-4453
J1808-2057
J1633-5015
J1603-5657
J1512-5759
J1733-3716
J1649-3805
J1801-2304
J1305-6455
J1750-3157
J1743-3150
J1820-1346
J1708-3426
J1759-3107
J1017-5621
J1534-5405
J1818-1422
J1807-2715
J1548-4927
J1802-2124
J1749-3002
J1707-4053
J1637-4816
J0905-4536
J1511-5414
J1527-5552
J1439-5501
J1716-4005
J1738-3211
J1649-4349
J1739-2903
J1820-1818
J1107-5907
J1827-0958
J1727-2951
J1816-1729
J1719-4006
J1610-5006
J1548-5607
J1708-3506
J1701-4533
J1809-2109
J1835-1020
J1319-6056
J1522-5829
J1653-3838
J1813-2621
J1832-1021
J1759-2922

In [49]:
#adding these stats to the existing dataframe.
#do not run this cell if using corrected fluxes. Instead, see cell below.
psrs_df['dof'] = dof
psrs_df['p_val'] = p_val
psrs_df['modulation'] = mod
psrs_df['modulation_err'] = mod_err
psrs_df['chi2'] = chi
psrs_df['reduced_chi2'] = chi_reduced

In [49]:
#run this cell instead if using this notebook with corrected fluxes. 
#This ensures that existing columns in the csv are not overwritten.
psrs_df['corrected_dof'] = dof
psrs_df['corrected_p_val'] = p_val
psrs_df['corrected_modulation'] = mod
psrs_df['corrected_modulation_err'] = mod_err
psrs_df['corrected_chi2'] = chi
psrs_df['corrected_reduced_chi2'] = chi_reduced

In [52]:
#re-saving to a csv. This csv now contains the properties of each VASTGalactic pulsar from ATNF along with the modulation, chi squared, reduced chi squared, p value, and degrees of freedom that we just computed
psrs_df.to_csv('paper_dfv1.csv')